# Quarantine Route Records

Routes bad records into quarantine table with proper classification and metadata.

**Purpose**: Capture failed records during pipeline processing (bronze/silver/semantic/warehouse stages)

**Usage**:
```python
route_to_quarantine(
    source_name="job_postings",
    failure_stage="SILVER",
    failed_rule="invalid_salary_range",
    severity="ERROR",
    bad_records_df=failed_df,
    batch_id=current_batch_id
)
```

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, current_timestamp, lit, to_json, struct, uuid
)
from datetime import datetime
import json

# Configuration
QUARANTINE_TABLE = "workspace.quarantine.quarantine_jobs"
DEFAULT_REPROCESS_STATUS = "PENDING"

In [0]:
def route_to_quarantine(
    source_name: str,
    failure_stage: str,
    failed_rule: str,
    severity: str,
    bad_records_df: DataFrame,
    batch_id: str,
    source_job_id: str = None,
    enterprise_job_id: str = None
) -> int:
    """
    Route bad records to quarantine table.
    
    Args:
        source_name: Source system name
        failure_stage: BRONZE, SILVER, SEMANTIC, WAREHOUSE
        failed_rule: Name of validation rule that failed
        severity: ERROR or WARNING
        bad_records_df: DataFrame containing failed records
        batch_id: Current batch identifier
        source_job_id: Optional source job ID
        enterprise_job_id: Optional enterprise job ID
        
    Returns:
        Count of records routed to quarantine
    """
    
    if bad_records_df.isEmpty():
        print(f"No records to quarantine for {source_name} - {failed_rule}")
        return 0
    
    # Convert records to JSON payload
    quarantine_df = bad_records_df.select(
        uuid().alias("quarantine_id"),
        lit(source_name).alias("source_name"),
        lit(source_job_id).alias("source_job_id"),
        lit(enterprise_job_id).alias("enterprise_job_id"),
        lit(failure_stage).alias("failure_stage"),
        lit(failed_rule).alias("failed_rule_name"),
        lit(severity).alias("severity"),
        to_json(struct("*")).alias("payload_json"),
        current_timestamp().alias("quarantined_at"),
        lit(batch_id).alias("batch_id"),
        lit(DEFAULT_REPROCESS_STATUS).alias("reprocess_status"),
        lit(None).cast("string").alias("reprocess_batch_id"),
        lit(None).cast("timestamp").alias("resolved_at")
    )
    
    # Append to quarantine table
    quarantine_df.write.mode("append").saveAsTable(QUARANTINE_TABLE)
    
    count = bad_records_df.count()
    print(f"✓ Routed {count} records to quarantine: {source_name} - {failed_rule} [{severity}]")
    
    return count

In [0]:
# Example usage - typically called from pipeline notebooks

# Simulate some failed records (replace with actual validation logic)
example_failed_df = spark.createDataFrame([
    ("J001", "E001", "Data Analyst", -50000, "2024-01-15"),
    ("J002", "E002", "Engineer", None, "2024-01-16")
], ["source_job_id", "enterprise_job_id", "title", "salary", "posted_date"])

# Route to quarantine
quarantined_count = route_to_quarantine(
    source_name="job_postings_linkedin",
    failure_stage="SILVER",
    failed_rule="salary_validation",
    severity="ERROR",
    bad_records_df=example_failed_df,
    batch_id="BATCH_20240601_001"
)

print(f"\nTotal records quarantined: {quarantined_count}")

In [0]:
def bulk_route_quarantine(failures_list: list, batch_id: str) -> dict:
    """
    Route multiple failure types to quarantine in one call.
    
    Args:
        failures_list: List of dicts with keys: source_name, failure_stage, 
                       failed_rule, severity, bad_records_df
        batch_id: Current batch identifier
        
    Returns:
        Dictionary with quarantine counts by rule
    """
    results = {}
    
    for failure in failures_list:
        count = route_to_quarantine(
            source_name=failure.get("source_name"),
            failure_stage=failure.get("failure_stage"),
            failed_rule=failure.get("failed_rule"),
            severity=failure.get("severity"),
            bad_records_df=failure.get("bad_records_df"),
            batch_id=batch_id,
            source_job_id=failure.get("source_job_id"),
            enterprise_job_id=failure.get("enterprise_job_id")
        )
        results[failure.get("failed_rule")] = count
    
    return results